## <span style="color:#4375c7">DAI</span>
***
*Course materials are for educational purposes only. Nothing contained herein should be considered investment advice or an opinion regarding the suitability of any security. For more information about this course, please contact us.*
***
## 6. Deep Learning — Transformers & GPT <a name="GPT"></a>

**Large Language Models (LLMs)** like ChatGPT are built on the **Transformer** architecture, which relies on a mechanism called **self-attention**. In this session we build a GPT (Generatively Pre-trained Transformer) from scratch using pure Python and NumPy, following Andrej Karpathy's [microGPT](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95) — a 200-line script that contains the complete algorithm that powers modern LLMs.

### Session contents:
1. **[Motivation](#motiv)**
2. **[Tokenization](#tok)**
3. **[The prediction game](#pred)**
4. **[From logits to probabilities — Softmax](#softmax)**
5. **[Measuring surprise — Cross-entropy loss](#loss)**
6. **[Token & positional embeddings](#embed)**
7. **[Self-attention](#attn)**
8. **[Multi-head attention](#mha)**
9. **[Transformer block](#block)**
10. **[Full GPT forward pass](#fwd)**
11. **[Training loop](#train)**
12. **[Inference & temperature](#infer)**
13. **[Keras implementation on economics text](#keras)**
14. **[Hands-on session](#ho)**
***

In [ ]:
!pip install -q -r https://raw.githubusercontent.com/firrm/DAI/main/requirements_colab.txt

import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import math
import random

### 1. Motivation <a id='motiv'></a>

The deep learning sessions so far covered:
- **Feedforward neural networks** — layers of weights, activation functions, backpropagation (session 5a)
- **Generative models** — VAEs and GANs that learn to sample from data distributions (session 5b)

The missing piece is **sequence modelling**: how do we generate text, one token at a time?

Before Transformers, sequence models were based on **Recurrent Neural Networks (RNNs)**, which process tokens one by one and carry a hidden state forward. This worked, but had two big problems:
1. **Vanishing gradients** — gradients shrink as they travel back through time, making it hard to learn long-range dependencies.
2. **No parallelism** — tokens must be processed sequentially, making training slow.

In 2017, Vaswani et al. introduced the **Transformer** ([Attention Is All You Need](https://arxiv.org/abs/1706.03762)), which solves both problems using **self-attention**: every token can directly attend to every other token in one parallel operation.

<br/>
<img src="https://raw.githubusercontent.com/firrm/DAI/main/assets/transformer_overview.png" alt="Transformer" style="width:500px;" onerror="this.style.display='none'"/>
<br/>

A **GPT** (Generatively Pre-trained Transformer) is a Transformer trained on the task of **next-token prediction**: given all previous tokens, predict the next one. From ChatGPT's perspective, your entire conversation is just a document, and the model is completing it token by token.

### 2. Tokenization <a id='tok'></a>

Neural networks work with **numbers**, not characters. We need a mapping from text → integers and back. The simplest possible tokenizer assigns one integer to each unique character.

We use Karpathy's microGPT dataset: 32,000 human first names (emma, olivia, ava …). Each name is a document. We add a special **BOS** (Beginning-of-Sequence) token that marks the start and end of each name.

In [ ]:
# Fetch the names dataset
import urllib.request
url = "https://raw.githubusercontent.com/karpathy/micrograd/master/names.txt"
try:
    with urllib.request.urlopen(url) as r:
        names_text = r.read().decode()
except Exception:
    # Fallback: embed a small sample
    names_text = "emma\nolivia\nava\nisabella\nsophia\ncharlotte\nmia\namelia\nharper\nevelyn\n"

names = [n.strip() for n in names_text.strip().split("\n") if n.strip()]
print(f"Dataset: {len(names):,} names")
print("First 10:", names[:10])

In [ ]:
# Build the vocabulary
chars = sorted(set("".join(names)))
vocab = ["BOS"] + chars          # BOS gets id=0, letters get ids 1-26
vocab_size = len(vocab)

stoi = {ch: i for i, ch in enumerate(vocab)}  # char → id
itos = {i: ch for ch, i in stoi.items()}       # id → char

BOS = stoi["BOS"]

print(f"Vocabulary ({vocab_size} tokens): {vocab}")
print()

def encode(name):
    """Encode a name as a list of integer ids, wrapped in BOS tokens."""
    return [BOS] + [stoi[c] for c in name] + [BOS]

def decode(ids):
    """Decode a list of integer ids back to a string."""
    return "".join(itos[i] for i in ids if i != BOS)

# Demonstrate
sample = "emma"
enc = encode(sample)
dec = decode(enc)
print(f"encode('{sample}') = {enc}")
print(f"decode({enc})      = '{dec}'")
assert dec == sample, "Round-trip failed!"
print("✓ Round-trip OK")

### 3. The prediction game <a id='pred'></a>

The training task is simple: **predict the next token, given all previous tokens**.

We slide a window through each tokenised name. At position $t$, the model sees tokens $0 \ldots t$ and must predict token $t+1$. For the name "emma" (encoded as `[BOS, e, m, m, a, BOS]`) that gives five training pairs:

| Context | Target |
|---|---|
| `[BOS]` | `e` |
| `[BOS, e]` | `m` |
| `[BOS, e, m]` | `m` |
| `[BOS, e, m, m]` | `a` |
| `[BOS, e, m, m, a]` | `BOS` |

This sliding window is exactly how ChatGPT trains — your conversation is the context, and the next token is the target.

In [ ]:
def make_training_pairs(name):
    """Return all (context, target) pairs for one name."""
    ids = encode(name)
    pairs = []
    for t in range(len(ids) - 1):
        context = ids[:t+1]
        target  = ids[t+1]
        pairs.append((context, target))
    return pairs

for ctx, tgt in make_training_pairs("emma"):
    ctx_str = [itos[i] for i in ctx]
    tgt_str = itos[tgt]
    print(f"  context={ctx_str:30s}  →  target='{tgt_str}'"  )

### 4. From logits to probabilities — Softmax <a id='softmax'></a>

At each position the model outputs **27 raw numbers** (one per possible next token). These **logits** can be anything — positive, negative, large, small. We need probabilities: positive numbers that sum to 1.

**Softmax** does this:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**Numerical stability**: subtracting the maximum logit before exponentiation prevents overflow (e.g. $e^{100} = \infty$ in float32), without changing the result:

$$p_i = \frac{e^{z_i - \max(z)}}{\sum_j e^{z_j - \max(z)}}$$

In [ ]:
def softmax(logits):
    """Numerically stable softmax."""
    logits = np.array(logits, dtype=float)
    logits -= logits.max()          # subtract max for numerical stability
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum()

# Demonstrate
logits = np.array([2.0, 1.0, 0.1, -1.0, 3.0])
probs  = softmax(logits)
print("Logits:", logits)
print("Probs: ", np.round(probs, 4))
print("Sum:   ", probs.sum())

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(len(logits)), logits, color="#4375c7")
axes[0].set_title("Logits"); axes[0].set_xlabel("Token id")
axes[1].bar(range(len(probs)), probs, color="#e87d4b")
axes[1].set_title("Probabilities after Softmax"); axes[1].set_xlabel("Token id")
plt.tight_layout(); plt.show()

### 5. Measuring surprise — Cross-entropy loss <a id='loss'></a>

How *wrong* was the prediction? We use **cross-entropy loss**:

$$\mathcal{L} = -\log p_{\text{correct}}$$

where $p_{\text{correct}}$ is the probability the model assigned to the correct next token.

- If $p_{\text{correct}} = 0.9$ → loss = $-\log 0.9 \approx 0.10$ (low — model was confident and correct)
- If $p_{\text{correct}} = 0.01$ → loss = $-\log 0.01 \approx 4.6$ (high — model was confident and *wrong*)

A random model assigning equal probability to all 27 tokens starts at $-\log(1/27) \approx 3.3$.

In [ ]:
def cross_entropy_loss(probs, target_id):
    """Cross-entropy loss for a single prediction."""
    return -np.log(probs[target_id] + 1e-12)  # eps avoids log(0)

# Visualise: loss as a function of p_correct
p_vals = np.linspace(0.01, 0.99, 200)
loss_vals = -np.log(p_vals)

plt.figure(figsize=(6, 3.5))
plt.plot(p_vals, loss_vals, color="#4375c7", lw=2)
plt.axhline(-np.log(1/27), color="gray", ls="--", label=f"Random baseline: {-np.log(1/27):.2f}")
plt.xlabel("Probability assigned to correct token $p$")
plt.ylabel("Loss  $-\log(p)$")
plt.title("Cross-entropy loss")
plt.legend(); plt.tight_layout(); plt.show()

### 6. Token & positional embeddings <a id='embed'></a>

A raw token id like `4` is just an index — the model cannot do arithmetic with it. Every token therefore **looks up a learned vector** in an embedding table. Think of it as each token having a personality of `d_model` numbers that the model tunes during training.

Position also matters: the letter "a" at position 0 plays a different role than "a" at position 4. A second embedding table indexes by position. The two embeddings are **added** together.

In microGPT:
- `d_model = 16` (embedding dimension)
- `context_length = 8` (maximum sequence length)
- Token embedding table: shape `(vocab_size, d_model)` = `(27, 16)`
- Position embedding table: shape `(context_length, d_model)` = `(8, 16)`

In [ ]:
# Hyperparameters (microGPT defaults)
d_model        = 16   # embedding / hidden dimension
context_length = 8    # maximum context window
n_heads        = 4    # number of attention heads
d_head         = d_model // n_heads  # dimension per head = 4
n_layers       = 1    # transformer layers (microGPT uses 1)

np.random.seed(42)

# Initialise embedding tables with small random values
W_token = np.random.randn(vocab_size, d_model) * 0.01      # (27, 16)
W_pos   = np.random.randn(context_length, d_model) * 0.01  # (8, 16)

def embed(token_ids):
    """
    Embed a sequence of token ids (length T) into shape (T, d_model).
    token_ids: list/array of ints, length T
    """
    T = len(token_ids)
    tok_emb = W_token[token_ids]          # (T, d_model)
    pos_emb = W_pos[:T]                   # (T, d_model)
    return tok_emb + pos_emb              # (T, d_model)

ids = encode("emma")[:-1]   # all but the last BOS
x = embed(ids)
print(f"Token ids:       {ids}")
print(f"Embedding shape: {x.shape}")  # (5, 16)
print(f"First row:\n{x[0].round(4)}")

### 7. Self-attention <a id='attn'></a>

Self-attention lets every token **gather information from previous tokens**. Each token produces three vectors from its embedding:

- **Query** $Q$ — "what am I looking for?"
- **Key** $K$ — "what do I contain?"
- **Value** $V$ — "what information do I offer if selected?"

The query at position $t$ is compared against all keys at positions $\le t$ via **dot product**. A high dot product means high relevance. Softmax turns these scores into attention weights:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}} + M\right) V$$

where $d_k$ is the key dimension and $M$ is a **causal mask** that sets future positions to $-\infty$ (so their softmax weight becomes 0 — the model cannot see the future).

In [ ]:
def attention(Q, K, V):
    """
    Scaled dot-product attention with causal masking.
    Q, K, V: arrays of shape (T, d_head)
    Returns: output of shape (T, d_head)
    """
    T, dk = Q.shape
    # Scaled dot-product scores
    scores = Q @ K.T / math.sqrt(dk)              # (T, T)

    # Causal mask: fill upper triangle with -inf
    mask = np.triu(np.full((T, T), -np.inf), k=1)
    scores = scores + mask                        # (T, T)

    # Softmax over the last axis (over keys)
    weights = np.apply_along_axis(softmax, axis=1, arr=scores)  # (T, T)

    return weights @ V                            # (T, d_head)

# Project embeddings to Q, K, V (one head for illustration)
np.random.seed(0)
W_q = np.random.randn(d_model, d_head) * 0.01
W_k = np.random.randn(d_model, d_head) * 0.01
W_v = np.random.randn(d_model, d_head) * 0.01

Q = x @ W_q   # (T, d_head)
K = x @ W_k
V = x @ W_v

out = attention(Q, K, V)
print("Attention output shape:", out.shape)

# Visualise attention weights
T = len(ids)
scores_raw = Q @ K.T / math.sqrt(d_head)
mask = np.triu(np.full((T, T), -np.inf), k=1)
weights = np.apply_along_axis(softmax, 1, scores_raw + mask)

labels = [itos[i] for i in ids]
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(T)); ax.set_xticklabels(labels, fontsize=11)
ax.set_yticks(range(T)); ax.set_yticklabels(labels, fontsize=11)
ax.set_title("Attention weights (1 head, 'emma')")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

### 8. Multi-head attention <a id='mha'></a>

With **multi-head attention** we run $h$ attention heads **in parallel**, each operating on a different $d_{head} = d_{model}/h$ dimensional slice of the embedding. Each head can learn a different pattern:
- Head 1: look at the most recent token
- Head 2: look at vowels
- Head 3: remember where the name started (BOS)

The outputs of all heads are **concatenated** and projected back to $d_{model}$:

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model, n_heads, rng=None):
        rng = rng or np.random
        d_head = d_model // n_heads
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head  = d_head
        # Weight matrices for Q, K, V for each head, and output projection
        self.W_q = [rng.randn(d_model, d_head) * 0.01 for _ in range(n_heads)]
        self.W_k = [rng.randn(d_model, d_head) * 0.01 for _ in range(n_heads)]
        self.W_v = [rng.randn(d_model, d_head) * 0.01 for _ in range(n_heads)]
        self.W_o = rng.randn(d_model, d_model) * 0.01  # output projection

    def forward(self, x):
        """x: (T, d_model) → output: (T, d_model)"""
        head_outputs = []
        for h in range(self.n_heads):
            Q = x @ self.W_q[h]
            K = x @ self.W_k[h]
            V = x @ self.W_v[h]
            head_outputs.append(attention(Q, K, V))   # (T, d_head) each
        concat = np.concatenate(head_outputs, axis=-1)  # (T, d_model)
        return concat @ self.W_o                         # (T, d_model)

np.random.seed(42)
mha = MultiHeadAttention(d_model, n_heads)
mha_out = mha.forward(x)
print("Multi-head attention output shape:", mha_out.shape)

### 9. Transformer block <a id='block'></a>

A **Transformer block** wraps multi-head attention and a feed-forward network (MLP) with two key stabilising techniques:

1. **Residual connections** (skip connections): the block computes $x \leftarrow x + \text{sublayer}(x)$. Without them, gradients would shrink to near zero through many layers and training would stall.

2. **RMSNorm** (Root-Mean-Square Normalisation): rescales each vector to have unit RMS before each sublayer. This prevents activations from exploding or vanishing.

The MLP is a two-layer network: project up to $4 \times d_{model}$ dimensions → ReLU → project back to $d_{model}$.

In [ ]:
def rms_norm(x, eps=1e-8):
    """Root-mean-square normalisation, applied row-wise."""
    rms = np.sqrt((x ** 2).mean(axis=-1, keepdims=True) + eps)
    return x / rms

class TransformerBlock:
    def __init__(self, d_model, n_heads, rng=None):
        rng = rng or np.random
        d_ff = 4 * d_model
        self.mha = MultiHeadAttention(d_model, n_heads, rng)
        self.W1  = rng.randn(d_model, d_ff)   * 0.01
        self.b1  = np.zeros(d_ff)
        self.W2  = rng.randn(d_ff,   d_model) * 0.01
        self.b2  = np.zeros(d_model)

    def forward(self, x):
        """x: (T, d_model) → (T, d_model)"""
        # Sub-layer 1: Multi-head self-attention + residual
        x = x + self.mha.forward(rms_norm(x))
        # Sub-layer 2: MLP + residual
        h = np.maximum(0, rms_norm(x) @ self.W1 + self.b1)  # ReLU
        x = x + h @ self.W2 + self.b2
        return x

np.random.seed(42)
block = TransformerBlock(d_model, n_heads)
block_out = block.forward(x)
print("Transformer block output shape:", block_out.shape)

### 10. Full GPT forward pass <a id='fwd'></a>

Putting it all together: the GPT processes a token sequence through the following pipeline for every position:

```
token ids  →  embed (token + position)
           →  transformer block(s)
           →  RMSNorm
           →  linear projection to vocab_size   (the "language model head")
           →  logits  →  softmax  →  probabilities
```

The output at position $t$ is a probability distribution over the next token, conditioned on all tokens $0 \ldots t$.

In [ ]:
class GPT:
    def __init__(self, vocab_size, context_length, d_model, n_heads, n_layers, rng=None):
        rng = rng or np.random
        self.vocab_size     = vocab_size
        self.context_length = context_length
        self.d_model        = d_model
        self.n_layers       = n_layers

        # Embedding tables
        self.W_token = rng.randn(vocab_size, d_model)     * 0.01
        self.W_pos   = rng.randn(context_length, d_model) * 0.01

        # Transformer blocks
        self.blocks = [TransformerBlock(d_model, n_heads, rng) for _ in range(n_layers)]

        # Output projection (language model head)
        self.W_lm = rng.randn(d_model, vocab_size) * 0.01

    def forward(self, token_ids):
        """
        token_ids: list of ints, length T (≤ context_length)
        Returns: logits of shape (T, vocab_size)
        """
        T = len(token_ids)
        # Embed
        x = self.W_token[token_ids] + self.W_pos[:T]   # (T, d_model)
        # Pass through transformer blocks
        for block in self.blocks:
            x = block.forward(x)
        # Final normalisation + project to vocabulary
        x = rms_norm(x)
        logits = x @ self.W_lm                          # (T, vocab_size)
        return logits

np.random.seed(42)
gpt = GPT(vocab_size, context_length, d_model, n_heads, n_layers)

ids = encode("emma")[:-1]   # input: BOS + e + m + m + a  (length 5)
logits = gpt.forward(ids)
print(f"Input ids:    {ids}")
print(f"Logits shape: {logits.shape}")   # (5, 27)
probs_last = softmax(logits[-1])
print(f"Predicted distribution over next token (after 'a'):")
for i, p in sorted(enumerate(probs_last), key=lambda x: -x[1])[:5]:
    print(f"  '{itos[i]}': {p:.4f}")

### 11. Training loop <a id='train'></a>

Training repeats the following for each name in the dataset:

1. **Forward pass**: compute logits for all positions in the name.
2. **Loss**: compute cross-entropy loss at each position; average over the name.
3. **Backward pass**: use autograd (or manual gradients) to compute $\partial \mathcal{L}/\partial \theta$ for every parameter $\theta$.
4. **Update**: adjust parameters with the **Adam** optimizer.

**Adam** maintains a running average of past gradients (momentum) and past squared gradients (adaptive learning rate). Parameters that receive consistent gradients take larger steps; oscillating parameters take smaller ones.

> *Note: a from-scratch Adam + autograd implementation requires several hundred lines of boilerplate. Below we show the training loop using **Keras**, which handles all of this for us. The algorithm is identical to microGPT.*

In [ ]:
# We demonstrate the loss curve for the pure-numpy GPT to show training dynamics.
# For a full trainable version, see section 13 (Keras).

# Quick loss demo: compute loss on one name with randomly initialised weights
def compute_name_loss(model, name):
    ids = encode(name)
    logits = model.forward(ids[:-1])   # input: all but last BOS
    targets = ids[1:]                  # target: shifted by one
    losses = [cross_entropy_loss(softmax(logits[t]), targets[t])
              for t in range(len(targets))]
    return np.mean(losses)

# Random baseline should be ≈ -log(1/27) ≈ 3.3
sample_names = names[:20]
losses = [compute_name_loss(gpt, n) for n in sample_names]
print(f"Mean loss (random init): {np.mean(losses):.3f}")
print(f"Expected random baseline: {-np.log(1/vocab_size):.3f}")

### 12. Inference & temperature <a id='infer'></a>

Once trained, **inference** works autoregressively:

1. Start with `[BOS]`.
2. Run the forward pass; take the logits at the last position.
3. Divide logits by **temperature** $\tau$, then apply softmax.
4. **Sample** one token from the distribution.
5. Append it to the context and repeat until BOS is sampled (name is done).

**Temperature** controls diversity:
- $\tau \to 0$: always pick the highest-probability token (greedy, "most average" output)
- $\tau = 1$: sample from the learned distribution
- $\tau > 1$: flatten the distribution → more diverse, potentially nonsensical

In [ ]:
def generate_name(model, temperature=1.0, max_len=20, seed=None):
    """Generate one name autoregressively."""
    rng = np.random.default_rng(seed)
    context = [BOS]
    for _ in range(max_len):
        ctx = context[-context_length:]   # truncate to context window
        logits = model.forward(ctx)
        logits_last = logits[-1] / temperature
        probs = softmax(logits_last)
        next_id = rng.choice(vocab_size, p=probs)
        if next_id == BOS:
            break
        context.append(next_id)
    return decode(context)

# With random weights the names will be gibberish — that's expected before training.
print("Generated names (untrained model, temperature=1.0):")
for i in range(8):
    print(" ", generate_name(gpt, temperature=1.0, seed=i))

print()
print("Effect of temperature on the distribution:")
logits_example = np.array([3.0, 1.5, 0.5, -0.5, -2.0])
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
for ax, tau in zip(axes, [0.5, 1.0, 2.0]):
    probs = softmax(logits_example / tau)
    ax.bar(range(len(probs)), probs, color="#4375c7")
    ax.set_title(f"temperature = {tau}")
    ax.set_xlabel("Token")
    ax.set_ylim(0, 1)
plt.suptitle("Softmax output at different temperatures", y=1.02)
plt.tight_layout(); plt.show()

### 13. Keras implementation on economics text <a id='keras'></a>

In practice we use frameworks like Keras/TensorFlow that provide automatic differentiation and GPU support. Below we build a small character-level GPT with Keras and train it on a short excerpt from a **Federal Reserve FOMC statement** — a real-world economics text.

The Keras model mirrors the numpy GPT exactly:
- `Embedding` layer for token + position embeddings
- `MultiHeadAttention` for self-attention (with causal masking)
- Feed-forward block with ReLU
- Dense output layer (language model head)

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers
import tensorflow as tf

# Short FOMC text corpus (public domain — official Federal Reserve statements)
corpus = """
The Federal Open Market Committee decided to maintain the target range for the federal funds rate.
The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run.
In support of these goals, the Committee decided to raise the target range for the federal funds rate.
Recent indicators suggest that economic activity has continued to expand at a modest pace.
Job gains have been robust in recent months, and the unemployment rate has remained low.
Inflation remains elevated, reflecting supply and demand imbalances related to the pandemic.
The Committee is highly attentive to inflation risks and is committed to returning inflation to its 2 percent objective.
The pace of future increases in the target range will take into account the cumulative tightening of monetary policy.
""".strip()

# Character-level tokenisation of the corpus
chars_corp = sorted(set(corpus))
vocab_c  = len(chars_corp)
s2i_c = {c: i for i, c in enumerate(chars_corp)}
i2s_c = {i: c for c, i in s2i_c.items()}

encoded_corpus = [s2i_c[c] for c in corpus]
print(f"Corpus: {len(corpus)} characters, {vocab_c} unique chars")

In [ ]:
# Build training sequences (sliding windows)
SEQ_LEN = 40   # context window

def make_sequences(enc, seq_len):
    X, y = [], []
    for i in range(len(enc) - seq_len):
        X.append(enc[i : i + seq_len])
        y.append(enc[i + 1 : i + seq_len + 1])
    return np.array(X), np.array(y)

X, y = make_sequences(encoded_corpus, SEQ_LEN)
print(f"Training sequences: X={X.shape}, y={y.shape}")

In [ ]:
# ── Keras causal GPT ──────────────────────────────────────────────
D_MODEL  = 64
N_HEADS  = 4
D_FF     = 256
N_LAYERS = 2

def causal_gpt(vocab_size, seq_len, d_model, n_heads, d_ff, n_layers):
    inputs = keras.Input(shape=(seq_len,), dtype="int32")

    # Token + position embeddings
    tok_emb = layers.Embedding(vocab_size, d_model)(inputs)                    # (B, T, d)
    pos_idx = tf.range(seq_len)
    pos_emb = layers.Embedding(seq_len, d_model)(pos_idx)                      # (T, d)
    x = tok_emb + pos_emb

    # Transformer blocks
    for _ in range(n_layers):
        # Self-attention with causal mask
        attn_out = layers.MultiHeadAttention(
            num_heads=n_heads, key_dim=d_model // n_heads, use_bias=False
        )(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + attn_out)

        # Feed-forward
        ff = layers.Dense(d_ff, activation="relu")(x)
        ff = layers.Dense(d_model)(ff)
        x = layers.LayerNormalization()(x + ff)

    # Language model head
    logits = layers.Dense(vocab_size)(x)    # (B, T, vocab_size)
    return keras.Model(inputs, logits)

model_keras = causal_gpt(vocab_c, SEQ_LEN, D_MODEL, N_HEADS, D_FF, N_LAYERS)
model_keras.summary()

In [ ]:
model_keras.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history = model_keras.fit(
    X, y,
    batch_size=32,
    epochs=200,
    verbose=0
)

# Plot training loss
plt.figure(figsize=(7, 3))
plt.plot(history.history["loss"], color="#4375c7")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss")
plt.title("Keras GPT — training on FOMC text")
plt.tight_layout(); plt.show()
print(f"Final loss: {history.history['loss'][-1]:.4f}")

In [ ]:
def generate_keras(model, seed_text, temperature=1.0, n_chars=200):
    """Autoregressively generate text from a Keras character-level GPT."""
    rng = np.random.default_rng(0)
    context = [s2i_c.get(c, 0) for c in seed_text[-SEQ_LEN:]]

    result = list(seed_text)
    for _ in range(n_chars):
        padded = context[-SEQ_LEN:]
        if len(padded) < SEQ_LEN:
            padded = [0] * (SEQ_LEN - len(padded)) + padded
        x_in = np.array([padded])
        logits = model.predict(x_in, verbose=0)[0, -1]   # logits for last position
        logits = logits / temperature
        probs  = softmax(logits)
        next_id = rng.choice(len(probs), p=probs)
        result.append(i2s_c[next_id])
        context.append(next_id)

    return "".join(result)

print("=== Generated text (temperature=0.5) ===")
print(generate_keras(model_keras, "The Committee", temperature=0.5, n_chars=150))
print()
print("=== Generated text (temperature=1.0) ===")
print(generate_keras(model_keras, "The Committee", temperature=1.0, n_chars=150))

### Session takeaways

| Concept | Key idea |
|---|---|
| **Tokenisation** | Map text → integers; simplest: one id per character |
| **Softmax** | Convert raw logits to probabilities; subtract max for numerical stability |
| **Cross-entropy loss** | $-\log p_{\text{correct}}$; punishes confident wrong predictions |
| **Embeddings** | Learned vectors for tokens and positions |
| **Self-attention** | Q/K/V dot-product; causal mask prevents future leakage |
| **Multi-head attention** | Parallel heads learn different relationship patterns |
| **Residual connections** | Skip connections prevent vanishing gradients in deep networks |
| **RMSNorm** | Stabilises activations by rescaling to unit RMS |
| **MLP block** | Each position "thinks" independently after attention |
| **Adam** | Adaptive, momentum-based gradient descent |
| **Temperature** | Controls diversity of generated samples |

The complete algorithm is only ~200 lines of Python. Between microGPT and ChatGPT, the *concepts* are identical — only scale changes: more data, bigger embeddings, more layers, GPUs.

### References

[1] Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS. https://arxiv.org/abs/1706.03762

[2] Karpathy, A. (2026). *microGPT*. https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95

[3] Growing SWE. (2026). *Visual walkthrough of microGPT*. https://growingswe.com/blog/microgpt

[4] Radford, A. et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI. https://openai.com/research/language-unsupervised

### 2. Hands-on session <a id='ho'></a>

See the [exercise notebook](../exercises/06_GPT_students%20version.ipynb) for practice tasks.